# Pathway coverage after GO:BP-prior retraining

CLAMPfull is retrained with the pinned GO:BP prior and evaluated against canonical npathways, and CellMarker with `clusterProfiler::enricher`, independently per database. Recovery uses FDR 0.05 throughout, recomputed here directly from each ORA's saved per-LV results.

💡 **Environment:** `clamp-analyses`

## Libraries

In [ ]:
suppressPackageStartupMessages({
    library(data.table)
    library(ggplot2)
    library(ggrepel)
    library(here)
})

stopifnot(
    nrow(fread(snakemake@input[["coverage_long"]])) > 0,
    nrow(fread(snakemake@input[["cross_dataset"]])) > 0,
    nrow(fread(snakemake@input[["panel_ready"]])) > 0
)

FDR <- 0.05

ora_root <- here(snakemake@config[["archs4"]][["coverage"]][["ora_root"]])
summary_paths <- list.files(ora_root, pattern = "^summary\\.csv$", recursive = TRUE, full.names = TRUE)
canonical <- "/[^/]+/rs[0-9]+/seed[0-9]+/[^./]+/[^./]+/summary\\.csv$"
summary_paths <- grep(canonical, summary_paths, value = TRUE)
if (length(summary_paths) == 0) {
    stop("No per-LV summary.csv files found under ", ora_root, " -- ORA outputs are missing.")
}

coverage <- rbindlist(lapply(summary_paths, function(spath) {
    s <- fread(spath)
    dir <- dirname(spath)
    enrich <- fread(file.path(dir, "enrichment.csv.gz"), select = c("ID", "p.adjust"))
    recovered <- if (nrow(enrich)) uniqueN(enrich[`p.adjust` < FDR, ID]) else 0L
    s[, `:=`(
        recovered_pathways = recovered,
        recovered_percent = 100 * recovered / eligible_pathways,
        fdr = FDR
    )]
    s
}), fill = TRUE)
setorder(coverage, dataset, database, fraction, model, seed)
cross <- coverage[fraction == 100]

model_colors <- unlist(snakemake@config[["archs4"]][["coverage"]][["colors"]][["model"]])
database_order <- c("reactome", "canonical", "cellmarker")
coverage[, database := factor(database, levels = database_order)]
coverage[, model := factor(model, levels = c("CLAMPfull", "CLAMPbase"))]
cross[, database := factor(database, levels = database_order)]
cross[, model := factor(model, levels = c("CLAMPfull", "CLAMPbase"))]

coverage[, .N, by = .(dataset, model, database)]

## Dataset-specific ORA universes and denominators

In [ ]:
coverage[, .(
    universe_size = unique(universe_size),
    eligible_pathways = unique(eligible_pathways),
    top_one_percent_genes = unique(query_target)
), by = .(dataset, database_label)]

## ARCHS4 coverage by sample fraction

In [ ]:
arch <- coverage[dataset == "archs4" & fraction < 100]
arch_seed <- arch[, .(
    recovered_pathways = sum(recovered_pathways),
    eligible_pathways = sum(eligible_pathways)
), by = .(fraction, model, seed)]
sample_labels <- arch[, .(n_samples = as.integer(median(n_samples))), by = fraction]
sample_labels[, label := sprintf("%s%%", fraction)]
arch_seed[, fraction_label := factor(
    fraction, levels = sample_labels$fraction, labels = sample_labels$label
)]

arch_labels <- arch_seed[, .(
    label_y = max(recovered_pathways),
    mean_recovered = mean(recovered_pathways),
    eligible_pathways = unique(eligible_pathways)
), by = .(model, fraction_label)]
arch_labels[, pct := 100 * mean_recovered / eligible_pathways]
arch_labels[, annotation := sprintf("%.0f (%.0f%%)", mean_recovered, pct)]

make_arch_plot <- function(model_name) {
    d <- arch_seed[model == model_name]
    lbl <- arch_labels[model == model_name]
    full_rng <- diff(range(d$recovered_pathways))
    lbl[, label_y := label_y + pmax(0.08 * full_rng, 1)]
    line <- d[, .(recovered_pathways = mean(recovered_pathways)), by = fraction_label]

    ggplot(d, aes(fraction_label, recovered_pathways, group = fraction_label)) +
        geom_boxplot(
            width = 0.5, outlier.shape = NA, alpha = 0.25, linewidth = 0.5,
            fill = model_colors[[model_name]], colour = model_colors[[model_name]]
        ) +
        geom_point(
            position = position_jitter(width = 0.07, height = 0),
            size = 2.1, alpha = 0.85, colour = model_colors[[model_name]]
        ) +
        geom_line(
            data = line, aes(group = 1),
            linetype = "dashed", linewidth = 0.5, colour = "#222222"
        ) +
        geom_text_repel(
            data = lbl, aes(y = label_y, label = annotation, group = NULL),
            colour = "black", size = 3.8, lineheight = 0.95,
            direction = "both", seed = 2, segment.size = 0.3, segment.alpha = 0.5,
            min.segment.length = 0.15, box.padding = 0.3, point.padding = 0.05,
            force = 1, force_pull = 0.5, max.time = 3, max.iter = 30000, max.overlaps = Inf
        ) +
        scale_y_continuous(expand = expansion(mult = c(0.05, 0.16))) +
        labs(
            x = "Training compendium coverage", y = "Recovered pathways (combined, 3 databases)",
            title = sprintf("%s - ARCHS4 coverage by sample fraction (FDR %.2f)", model_name, FDR)
        ) +
        theme_classic(base_size = 15) +
        theme(
            axis.text.x = element_text(size = 11),
            plot.title = element_text(face = "bold", size = 14)
        )
}

options(repr.plot.width = 11, repr.plot.height = 6.5)
make_arch_plot("CLAMPfull")
make_arch_plot("CLAMPbase")

## Full-data comparison

In [ ]:
dataset_names <- c(archs4 = "ARCHS4", gtex = "GTEx", recount2 = "recount2")
dataset_colors <- unlist(snakemake@config[["archs4"]][["coverage"]][["colors"]][["dataset"]])
REFERENCE_SEED <- 1

cross_one <- cross[
    (model == "CLAMPfull" & seed == REFERENCE_SEED) |
    (model == "CLAMPbase" & dataset == "archs4" & seed == REFERENCE_SEED) |
    (model == "CLAMPbase" & dataset != "archs4")
]
cross_one_sum <- cross_one[, .(recovered_pathways = sum(recovered_pathways)), by = .(dataset, model)]

make_cross_plot <- function(model_name) {
    d <- cross_one_sum[model == model_name]
    setorder(d, recovered_pathways)
    d[, dataset_label := factor(dataset, levels = dataset, labels = dataset_names[dataset])]
    ggplot(d, aes(dataset_label, recovered_pathways, colour = dataset_label)) +
        geom_point(shape = 18, size = 7) +
        geom_text(aes(label = recovered_pathways), vjust = -1.1, colour = "black", size = 4.5) +
        scale_colour_manual(values = dataset_colors, guide = "none") +
        scale_y_continuous(expand = expansion(mult = c(0.08, 0.15))) +
        labs(
            x = NULL, y = "Recovered pathways (combined, 3 databases)",
            title = sprintf("%s - full-data comparison across compendia (FDR %.2f)", model_name, FDR),
            caption = "One model per dataset (seed 1 where seeds exist)."
        ) +
        theme_classic(base_size = 15) +
        theme(
            axis.text.x = element_text(size = 12),
            plot.title = element_text(face = "bold", size = 14),
            plot.caption = element_text(size = 9, hjust = 0, colour = "#555555")
        )
}

options(repr.plot.width = 8, repr.plot.height = 6)
make_cross_plot("CLAMPfull")
make_cross_plot("CLAMPbase")

### GTEx / recount2 by database

In [ ]:
cross_one[dataset %in% c("gtex", "recount2"), .(
    recovered_pathways, eligible_pathways = unique(eligible_pathways), pct = round(recovered_percent, 1)
), by = .(dataset, database_label, model)][order(dataset, model, database_label)]